In [1]:
import os
os.environ["GDAL_DATA"] = r"C:\Users\WeiKitPhang\.conda\envs\python_base_1\Library\share\gdal"


import ee
import geemap
import geopandas as gpd
import pandas as pd

# --------------------------------------------------
# Initialize Earth Engine
# --------------------------------------------------
ee.Authenticate()
ee.Initialize()



In [2]:
shapefile_path = r"C:\Users\WeiKitPhang\Documents\UM\Writing\Paper 21 (Gua Musang Preliminary)\GIS\GM_shapefile.shp"

gdf = gpd.read_file(shapefile_path)

# Ensure CRS is correct
if gdf.crs is None:
    gdf.set_crs(epsg=4326, inplace=True)
else:
    gdf = gdf.to_crs(epsg=4326)

# Convert to Earth Engine object
roi = geemap.geopandas_to_ee(gdf)

print("Shapefile loaded successfully. CRS:", gdf.crs)

Shapefile loaded successfully. CRS: EPSG:4326


In [3]:
# ==========================================================
# 3. LOAD EXCEL → CONVERT TO EE FEATURECOLLECTION
# ==========================================================
excel_path = r"C:\Users\WeiKitPhang\Documents\UM\Writing\Paper 21 (Gua Musang Preliminary)\Data_processing\Gua Musang case data.xlsx"

df = pd.read_excel(excel_path, sheet_name="Week_data")

df["start_dt"] = pd.to_datetime(df["start_dt"])
df["end_dt"] = pd.to_datetime(df["end_dt"])

# Filter 2012–2023 only
df = df[(df["start_dt"] >= "2012-01-01") & (df["end_dt"] <= "2023-12-31")]

# Convert to EE FeatureCollection
features = []
for _, row in df.iterrows():
    features.append(ee.Feature(None, {
        "start_dt": row["start_dt"].strftime("%Y-%m-%d"),
        "end_dt": row["end_dt"].strftime("%Y-%m-%d")
    }))

weeks_fc = ee.FeatureCollection(features)

print(df.columns)
print(df.dtypes)

df.head()

Index(['epid_wk', 'start_dt', 'end_dt', 'case_count', 'case_count_l1',
       'case_count_l2', 'case_count_l3', 'precipitation', 'temp_2m',
       'dewpoint_2m', 'rh', 'pop_smooth', 'nino4_smooth', 'soi_smooth',
       'mei_smooth', 'mjo_romi'],
      dtype='object')
epid_wk                   int64
start_dt         datetime64[ns]
end_dt           datetime64[ns]
case_count                int64
case_count_l1           float64
case_count_l2           float64
case_count_l3           float64
precipitation           float64
temp_2m                 float64
dewpoint_2m             float64
rh                      float64
pop_smooth              float64
nino4_smooth            float64
soi_smooth              float64
mei_smooth              float64
mjo_romi                float64
dtype: object


,epid_wk,start_dt,end_dt,case_count,case_count_l1,case_count_l2,case_count_l3,precipitation,temp_2m,dewpoint_2m,rh,pop_smooth,nino4_smooth,soi_smooth,mei_smooth,mjo_romi
0,1,2012-01-01,2012-01-07,0,NaN,NaN,NaN,12.193868,22.743724,20.019748,84.615742,92267.867995,27.371389,1.238931,-1.127599,1.263833
1,2,2012-01-08,2012-01-14,1,0.0,NaN,NaN,27.072916,21.603099,20.543937,93.697011,92295.045862,27.344722,1.134913,-1.069003,1.273590
2,3,2012-01-15,2012-01-21,0,1.0,0.0,NaN,23.648809,23.101872,21.385557,90.071088,92322.223958,27.321981,1.030957,-1.010461,0.993704
3,4,2012-01-22,2012-01-28,3,0.0,1.0,0.0,20.649891,22.609929,19.398970,82.079763,92349.402670,27.305940,0.927549,-0.952063,1.443737
4,5,2012-01-29,2012-02-04,0,3.0,0.0,1.0,3.729955,22.771412,20.740097,88.318599,92376.582567,27.299368,0.825330,-0.893937,1.590466


In [5]:
# ==========================================================
# 5. CLOUD MASKING (QA_PIXEL)
# ==========================================================
def mask_landsat(img):

    qa = img.select("QA_PIXEL")

    cloud = 1 << 3
    shadow = 1 << 4
    snow = 1 << 5

    mask = qa.bitwiseAnd(cloud).eq(0) \
        .And(qa.bitwiseAnd(shadow).eq(0)) \
        .And(qa.bitwiseAnd(snow).eq(0))

    return img.updateMask(mask)

In [6]:
# Define per-sensor functions
# THEN harmonize bands
def harmonize_l57(img):
    return img.select(
        ["SR_B2", "SR_B4", "SR_B5"],
        ["GREEN", "NIR", "SWIR1"]
    )

def harmonize_l8(img):
    return img.select(
        ["SR_B3", "SR_B5", "SR_B6"],
        ["GREEN", "NIR", "SWIR1"]
    )

In [7]:
# ==========================================================
# DATASETS
# =========================================================
# PM2.5 proxy (MERRA-2)
merra2 = ee.ImageCollection("NASA/GSFC/MERRA/aer/2")

# Landsat collection (5/7/8)
landsat5 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2").map(mask_landsat).map(harmonize_l57)

landsat7 = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2").map(mask_landsat).map(harmonize_l57)

landsat8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").map(mask_landsat).map(harmonize_l8)

landsat = landsat5.merge(landsat7).merge(landsat8)

In [8]:
# NDVI and NDWI3 indixes

def add_indices(img):

    ndvi = img.normalizedDifference(["NIR","GREEN"]).rename("NDVI")

    ndwi3 = img.normalizedDifference(["GREEN","SWIR1"]).rename("NDWI3")

    return img.addBands(ndvi).addBands(ndwi3)

landsat_indices = landsat.map(add_indices)

In [9]:
# ==========================================================
# 7. NDVI + NDWI3 (MNDWI)
# ==========================================================
def add_indices(img):

    ndvi = img.normalizedDifference(
        ["NIR", "GREEN"]
    ).rename("NDVI")

    ndwi3 = img.normalizedDifference(
        ["GREEN", "SWIR1"]
    ).rename("NDWI3")

    return img.addBands(ndvi).addBands(ndwi3)

landsat_indices = landsat.map(add_indices)


In [10]:
# ==========================================================
# 9. WEEKLY FUNCTION
# ==========================================================
def compute_week(feature):

    start = ee.Date(feature.get("start_dt"))
    end = ee.Date(feature.get("end_dt"))

    # -------------------------
    # 🌫️ TOTAL PM2.5 (MERRA-2 aerosol sum)
    # -------------------------
    aero = merra2.filterDate(start, end.advance(1, "day")).mean()

    pm25 = aero.select([
        "BCSMASS",
        "OCSMASS",
        "SO4SMASS",
        "DUSMASS25",
        "SSSMASS25"
    ])

    bc = pm25.select("BCSMASS")
    oc = pm25.select("OCSMASS")
    so4 = pm25.select("SO4SMASS").multiply(1.375)
    dust = pm25.select("DUSMASS25")
    salt = pm25.select("SSSMASS25")

    pm25_final = bc.add(oc).add(so4).add(dust).add(salt) \
        .multiply(1e9) \
        .rename("PM25")

    # -------------------------
    # 🌿 NDVI (27-day backward window)
    # -------------------------
    ndvi = landsat_indices \
        .filterDate(end.advance(-45, "day"), end.advance(1, "day")) \
        .select("NDVI") \
        .median()

    # -------------------------
    # 🌊 NDWI3 (MNDWI)
    # -------------------------
    ndwi3 = landsat_indices \
        .filterDate(end.advance(-45, "day"), end.advance(1, "day")) \
        .select("NDWI3") \
        .median()

    # -------------------------
    # STACK VARIABLES
    # -------------------------
    img = pm25_final.addBands(ndvi).addBands(ndwi3)

    stats = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi.geometry(),
        scale=1000,
        maxPixels=1e13
    )

    return feature.set({
        "PM25": stats.get("PM25"),
        "NDVI": stats.get("NDVI"),
        "NDWI3": stats.get("NDWI3")
    })

In [11]:
# ==========================================================
# 8. APPLY FUNCTION
# ==========================================================
result_fc = weeks_fc.map(compute_week)

In [12]:
# ==========================================================
# 9. EXPORT
# ==========================================================
task = ee.batch.Export.table.toDrive(
    collection=result_fc,
    description="weekly_PM25_NDVI_NDWI3_2012_2023_FINAL",
    fileFormat="CSV"
)

task.start()

print("Export started. Check Earth Engine Tasks tab.")

Export started. Check Earth Engine Tasks tab.
